In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../dataset/Dataset.csv")

In [3]:
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
df.shape

(9551, 21)

In [5]:
df.columns

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   str    
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   str    
 4   Address               9551 non-null   str    
 5   Locality              9551 non-null   str    
 6   Locality Verbose      9551 non-null   str    
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   str    
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   str    
 12  Has Table booking     9551 non-null   str    
 13  Has Online delivery   9551 non-null   str    
 14  Is delivering now     9551 non-null   str    
 15  Switch to order menu  9551 non-n

In [7]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df = df.dropna(subset=["Cuisines"])

In [10]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                0
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [11]:
df["Cuisines"] = df["Cuisines"].str.split(",").str[0]

In [12]:
df["Cuisines"].head()

0      French
1    Japanese
2     Seafood
3    Japanese
4    Japanese
Name: Cuisines, dtype: object

In [13]:
df = df[[
        "Restaurant Name",
        "City",
        "Cuisines",
        "Price range",
        "Average Cost for two",
        "Aggregate rating",
        "Votes"]]

In [14]:
df.head()

,Restaurant Name,City,Cuisines,Price range,Average Cost for two,Aggregate rating,Votes
0,Le Petit Souffle,Makati City,French,3,1100,4.8,314
1,Izakaya Kikufuji,Makati City,Japanese,3,1200,4.5,591
2,Heat - Edsa Shangri-La,Mandaluyong City,Seafood,4,4000,4.4,270
3,Ooma,Mandaluyong City,Japanese,4,1500,4.9,365
4,Sambo Kojin,Mandaluyong City,Japanese,4,1500,4.8,229


In [15]:
feature_cols = [
    "City",
    "Cuisines",
    "Price range"
]

In [16]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_features = encoder.fit_transform(df[feature_cols])

In [17]:
encoded_features.shape

(9542, 263)

In [18]:
similarity = cosine_similarity(encoded_features)

In [19]:
similarity.shape

(9542, 9542)

In [32]:
def recommend_restaurant(cuisine, price_range, city=None, top_n=5):

    recommendations = df.copy()

    recommendations = recommendations[
        recommendations["Cuisines"] == cuisine
    ]

    recommendations = recommendations[
        recommendations["Price range"] == price_range
    ]

    if city is not None:
        recommendations = recommendations[
            recommendations["City"] == city
        ]

    recommendations = recommendations.sort_values(
        by=["Aggregate rating", "Votes"],
        ascending=False
    )

    return recommendations[
        [
            "Restaurant Name",
            "City",
            "Cuisines",
            "Price range",
            "Aggregate rating",
            "Votes"
        ]
    ].head(top_n)

In [33]:
recommend_restaurant(
    cuisine="North Indian",
    price_range=2
)

,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes
2456,Aman Chicken,Ludhiana,North Indian,2,4.6,196
6426,Food Scouts,New Delhi,North Indian,2,4.6,61
4206,Midnight Hunger Hub,New Delhi,North Indian,2,4.5,50
818,Palmshore,Chennai,North Indian,2,4.4,645
9194,Saffron Mantra,Secunderabad,North Indian,2,4.4,494


In [34]:
recommend_restaurant(
    cuisine="Chinese",
    price_range=3
)

,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes
123,Miyabi Kyoto Japanese Steak House,Augusta,Chinese,3,4.6,717
1222,Yum Yum Cha,Gurgaon,Chinese,3,4.5,407
5,Din Tai Fung,Mandaluyong City,Chinese,3,4.4,336
2285,The Woking Mama,Guwahati,Chinese,3,4.4,129
9259,Mekong - Hotel GreenPark,Vizag,Chinese,3,4.4,73


In [35]:
recommend_restaurant(
    cuisine="Italian",
    price_range=2,
    city="New Delhi"
)

,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes
3705,Sinyora's,New Delhi,Italian,2,4.0,51
5931,Pizza Hut Delivery,New Delhi,Italian,2,3.9,189
3243,NYC.PIE,New Delhi,Italian,2,3.8,306
4945,Play Pizza,New Delhi,Italian,2,3.8,270
4339,Starvin' Marvin,New Delhi,Italian,2,3.8,170


In [36]:
recommendation_system = {
    "data": df
}


In [37]:
import pickle

pickle.dump(
    recommendation_system,
    open("../models/restaurant_recommendation.pkl", "wb")
)